In [ ]:
# DNAnexus RAP (PySpark): admission-level acute coronary events (I21–I24, primary dx only)
# + death registry acute coronary deaths (I21–I24 anywhere in causes)
#
# Output: a CSV-like folder (Spark writes a directory) with one row per participant-event.
# Columns include: eid, event_date, source, admission_id (if inpatient), icd10_set, n_rows_in_admission

import dxpy, dxdata
import subprocess
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
import re, os, glob, shutil

In [ ]:
spark = SparkSession.builder.getOrCreate()

DB = "database_XXXXXXXXXXXXXXXXXXXXXXXX__appXXXXX_XXXXXXXXXXXXXX"

In [ ]:
hesin      = spark.table(f"`{DB}`.hesin")
hesin_diag = spark.table(f"`{DB}`.hesin_diag")

h = hesin.alias("h")
d = hesin_diag.alias("d")

# Join on eid + ins_index
hd = h.join(
    d,
    on=[F.col("h.eid") == F.col("d.eid"),
        F.col("h.ins_index") == F.col("d.ins_index")],
    how="inner"
)

# Primary dx: hesin_diag.level is STRING in your schema -> compare to "1" (or cast)
is_primary_dx = (F.col("d.level") == F.lit("1"))
# alternatively:
# is_primary_dx = (F.col("d.level").cast("int") == F.lit(1))

# Acute coronary I21–I24
icd = F.upper(F.col("d.diag_icd10"))
is_acute_coronary = icd.rlike(r"^I2[1-4]")

# Event date anchor (admidate exists and is already a date)
event_date = F.coalesce(F.col("h.admidate"), F.col("h.epistart"))

# Build row-level set
inpatient_rows = (
    hd
    .filter(is_primary_dx & is_acute_coronary)
    .withColumn("event_date", F.to_date(event_date))
    .filter(F.col("event_date").isNotNull())
    .select(
        F.col("h.eid").alias("eid"),
        F.col("h.spell_index").alias("spell_index"),
        F.col("h.ins_index").alias("ins_index"),
        F.col("event_date"),
        F.col("d.diag_icd10").alias("icd10"),
        F.col("d.level").alias("diag_pos"),
        F.col("h.admidate").alias("admidate"),
        F.col("h.epistart").alias("epistart"),
        F.col("h.epiend").alias("epiend"),
        F.col("h.disdate").alias("disdate"),
    )
)

# Collapse to admission/spell level (one row per eid x spell_index)
inpatient_events = (
    inpatient_rows
    .groupBy("eid", "spell_index")
    .agg(
        F.min("event_date").alias("event_date"),
        F.collect_set("icd10").alias("icd10_set"),
        F.count("*").alias("n_rows_in_admission"),
        F.collect_set("ins_index").alias("ins_index_set"),
    )
    .withColumnRenamed("spell_index", "admission_id")  # keep your downstream naming
    .withColumn("source", F.lit("inpatient_primary_I21_I24"))
)

In [ ]:
inpatient_events_day = (
    inpatient_events
    .groupBy("eid", "event_date")
    .agg(
        F.collect_set("admission_id").alias("admission_id"),
        F.array_distinct(F.flatten(F.collect_list("icd10_set"))).alias("icd10_set"),
        F.sum("n_rows_in_admission").alias("n_rows_in_admission"),
        F.array_distinct(F.flatten(F.collect_list("ins_index_set"))).alias("ins_index_set")
    )
    .withColumn("source", F.lit("inpatient_primary_I21_I24_dedup_by_day"))
)

In [ ]:
inpatient_events_day.show(20, truncate=False)

In [ ]:
death       = spark.table(f"`{DB}`.death")
death_cause = spark.table(f"`{DB}`.death_cause")

# Date of death per eid
dd = (
    death
    .select(
        F.col("eid").alias("eid"),
        F.col("date_of_death").alias("event_date")
    )
    .filter(F.col("event_date").isNotNull())
)

# Cause rows filtered to I21–I24
dc = (
    death_cause
    .select(
        F.col("eid").alias("eid"),
        F.col("level").alias("cause_level"),
        F.upper(F.col("cause_icd10")).alias("icd10")
    )
    .filter(F.col("icd10").rlike(r"^I2[1-4]"))  # I21–I24
)

# Join and collapse to one row per person-date (date_of_death is unique per eid)
death_joined = dd.join(dc, on="eid", how="inner")

death_events = (
    death_joined
    .groupBy("eid", "event_date")
    .agg(
        F.collect_set("icd10").alias("icd10_set"),
        F.collect_set("cause_level").alias("cause_level_set"),
        F.count("*").alias("n_cause_rows")
    )
    .withColumn("source", F.lit("death_I21_I24_any_cause"))
    .withColumn("admission_id", F.lit(None).cast("string"))
    .withColumn("n_rows_in_admission", F.lit(None).cast("long"))
    .withColumn("ins_index_set", F.array().cast("array<long>"))
)

In [ ]:
# Optionally dedup by day first (recommended if you saw same-day duplicates)
inpatient_final = inpatient_events_day

inpatient_out = (
    inpatient_events_day
    .withColumn("admission_id", F.col("admission_id").cast("string"))  # <-- key tweak
    .withColumn("n_cause_rows", F.lit(None).cast("long"))
    .withColumn("cause_level_set", F.lit(None).cast("array<long>"))
)

death_out = (
    death_events
    .withColumn("admission_id", F.lit(None).cast("string"))
    .withColumn("n_rows_in_admission", F.lit(None).cast("long"))
    .withColumn("ins_index_set", F.array().cast("array<long>"))
)

cols = [
    "eid",
    "event_date",
    "source",
    "admission_id",
    "icd10_set",
    "n_rows_in_admission",
    "ins_index_set",
    "n_cause_rows",
    "cause_level_set",
]

events_all = inpatient_out.select(*cols).unionByName(death_out.select(*cols))

# Summary
events_all.groupBy("source").agg(
    F.count("*").alias("n_events"),
    F.countDistinct("eid").alias("n_people"),
    F.min("event_date").alias("min_date"),
    F.max("event_date").alias("max_date"),
).show(truncate=False)

events_csv = (
    events_all
    # arrays -> pipe-delimited strings (pick any delimiter you like)
    .withColumn("icd10_set", F.concat_ws("|", F.col("icd10_set")))
    .withColumn("ins_index_set", F.when(F.col("ins_index_set").isNull(), F.lit(None))
                .otherwise(F.concat_ws("|", F.col("ins_index_set"))))
    .withColumn("admission_id", F.when(F.col("admission_id").isNull(), F.lit(None))
                .otherwise(F.concat_ws("|", F.col("admission_id"))))
    .withColumn("cause_level_set", F.when(F.col("cause_level_set").isNull(), F.lit(None))
                .otherwise(F.concat_ws("|", F.col("cause_level_set"))))
)

# Export
cwd = os.getcwd()
print("Current working directory:", cwd)

(
    events_csv
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(cwd)
)

final_path = os.path.join(cwd, "ukb_acute_coronary_events_inpatient_plus_death.csv.gz")

# events_csv should already have arrays serialised to strings
pdf = events_csv.toPandas()
pdf.to_csv(final_path, index=False, compression='gzip')

print("Wrote:", final_path, "rows:", len(pdf))

In [ ]:
project_id = dxpy.PROJECT_CONTEXT_ID
cmd = f"dx upload ukb_acute_coronary_events_inpatient_plus_death.csv.gz --path {project_id}:/path/to/scratch/"

returned_value = subprocess.call(cmd, shell=True)  # returns the exit code in unix
print('Returned value:', returned_value)

In [ ]:
hesin      = spark.table(f"`{DB}`.hesin")
hesin_oper = spark.table(f"`{DB}`.hesin_oper")

h = hesin.alias("h")
o = hesin_oper.alias("o")

# Join on eid + ins_index (your schema supports this directly)
ho = h.join(
    o,
    on=[F.col("h.eid") == F.col("o.eid"),
        F.col("h.ins_index") == F.col("o.ins_index")],
    how="inner"
)

# Use OPCS-4 codes in oper4 (string)
opcs4 = F.upper(F.col("o.oper4"))

# PCI / CABG code families (OPCS-4)
is_pci  = opcs4.rlike(r"^K49") | opcs4.rlike(r"^K50") | opcs4.rlike(r"^K75")
is_cabg = opcs4.rlike(r"^K4[0-4]")  # K40–K44
is_revasc = is_pci | is_cabg

# Procedure date: prefer opdate when present, else fall back to admission/episode start
event_date = F.coalesce(F.col("o.opdate"), F.col("h.admidate"), F.col("h.epistart"))

proc_rows = (
    ho
    .filter(opcs4.isNotNull() & is_revasc)
    .withColumn("event_date", F.to_date(event_date))
    .filter(F.col("event_date").isNotNull())
    .withColumn(
        "proc_type",
        F.when(is_cabg, F.lit("CABG"))
         .when(is_pci,  F.lit("PCI"))
         .otherwise(F.lit("OTHER"))
    )
    .select(
        F.col("h.eid").alias("eid"),
        F.col("event_date"),
        F.col("h.spell_index").alias("spell_index"),
        F.col("h.ins_index").alias("ins_index"),
        F.col("o.arr_index").alias("arr_index"),
        F.col("o.level").alias("op_level"),
        F.col("o.opdate").alias("opdate"),
        F.col("o.oper4").alias("oper4"),
        F.col("proc_type"),
    )
)

# Collapse to spell-level
proc_spell = (
    proc_rows
    .groupBy("eid", "spell_index")
    .agg(
        F.min("event_date").alias("event_date"),
        F.collect_set("oper4").alias("opcs4_set"),
        F.collect_set("proc_type").alias("proc_type_set"),
        F.collect_set("ins_index").alias("ins_index_set"),
        F.count("*").alias("n_rows_in_admission"),
        F.min("opdate").alias("min_opdate"),
        F.max("opdate").alias("max_opdate"),
    )
    .withColumnRenamed("spell_index", "admission_id")
    .withColumn("source", F.lit("inpatient_opcs_PCI_CABG_spell"))
)

# Dedup to one row per (eid, event_date); keep admission_id column name (now array<long>)
proc_day = (
    proc_spell
    .groupBy("eid", "event_date")
    .agg(
        F.collect_set("admission_id").alias("admission_id"),  # array<long>
        F.array_distinct(F.flatten(F.collect_set("opcs4_set"))).alias("opcs4_set"),
        F.array_distinct(F.flatten(F.collect_set("proc_type_set"))).alias("proc_type_set"),
        F.array_distinct(F.flatten(F.collect_set("ins_index_set"))).alias("ins_index_set"),
        F.sum("n_rows_in_admission").alias("n_rows_in_admission"),
        F.min("min_opdate").alias("min_opdate"),
        F.max("max_opdate").alias("max_opdate"),
    )
    .withColumn("source", F.lit("inpatient_opcs_PCI_CABG_dedup_by_day"))
)

# Sanity checks
proc_day.groupBy("source").agg(
    F.count("*").alias("n_events"),
    F.countDistinct("eid").alias("n_people"),
    F.min("event_date").alias("min_date"),
    F.max("event_date").alias("max_date"),
).show(truncate=False)

proc_day.show(20, truncate=False)


In [ ]:
proc_csv = (
    proc_day
    .withColumn("admission_id",   F.concat_ws("|", F.col("admission_id")))    # array<long> -> string
    .withColumn("opcs4_set",      F.concat_ws("|", F.col("opcs4_set")))       # array<string> -> string
    .withColumn("proc_type_set",  F.concat_ws("|", F.col("proc_type_set")))   # array<string> -> string
    .withColumn("ins_index_set",  F.concat_ws("|", F.col("ins_index_set")))  # array<long> -> string
)

# Write to the same directory as the notebook/script (driver-local)
cwd = os.getcwd()
out_path = os.path.join(cwd, "ukb_pci_cabg_events.csv.gz")

pdf = proc_csv.toPandas()
pdf.to_csv(out_path, index=False, compression='gzip')

print("Wrote:", out_path, "rows:", len(pdf))

In [ ]:
cmd = f"dx upload ukb_pci_cabg_events.csv.gz --path {project_id}:/path/to/scratch/"

returned_value = subprocess.call(cmd, shell=True)  # returns the exit code in unix
print('Returned value:', returned_value)